In [2]:
import psycopg2

#задание 3
conn = psycopg2.connect(
    dbname='demo',
    user='postgres',
    password='postgres',
    host='localhost',
    port='5433'   # ← ВАЖНО! НЕ 5432
)

cur = conn.cursor()

cur.execute("""
SELECT DISTINCT fare_conditions
FROM bookings.ticket_flights;
""")

print(cur.fetchall())

[('Business',), ('Comfort',), ('Economy',)]


In [5]:
#задание 1
cur.execute("""
SELECT table_name, column_name
FROM information_schema.columns
WHERE table_schema = 'bookings'
ORDER BY table_name, ordinal_position;
""")

print(cur.fetchall())

[('aircrafts', 'aircraft_code'), ('aircrafts', 'model'), ('aircrafts', 'range'), ('aircrafts_data', 'aircraft_code'), ('aircrafts_data', 'model'), ('aircrafts_data', 'range'), ('airports', 'airport_code'), ('airports', 'airport_name'), ('airports', 'city'), ('airports', 'coordinates'), ('airports', 'timezone'), ('airports_data', 'airport_code'), ('airports_data', 'airport_name'), ('airports_data', 'city'), ('airports_data', 'coordinates'), ('airports_data', 'timezone'), ('boarding_passes', 'ticket_no'), ('boarding_passes', 'flight_id'), ('boarding_passes', 'boarding_no'), ('boarding_passes', 'seat_no'), ('bookings', 'book_ref'), ('bookings', 'book_date'), ('bookings', 'total_amount'), ('flights', 'flight_id'), ('flights', 'flight_no'), ('flights', 'scheduled_departure'), ('flights', 'scheduled_arrival'), ('flights', 'departure_airport'), ('flights', 'arrival_airport'), ('flights', 'status'), ('flights', 'aircraft_code'), ('flights', 'actual_departure'), ('flights', 'actual_arrival'), (

aircrafts / aircrafts_data
Содержит информацию о самолётах (код, модель, дальность полёта).

airports / airports_data
Содержит данные об аэропортах (код, название, город, координаты, часовой пояс).

boarding_passes
Хранит посадочные талоны (номер билета, рейс, место).

bookings
Информация о бронированиях (номер, дата, общая сумма).

flights
Основная таблица рейсов (номер рейса, аэропорты, время, статус, самолёт).

flights_v
Представление (view) с расширенной информацией о рейсах (включая города, длительность и локальное время).

routes
Маршруты перелётов (откуда → куда, длительность, дни недели).

seats
Места в самолётах (номер места, класс обслуживания).

ticket_flights
Связь билетов и рейсов + стоимость и тариф.

tickets
Информация о билетах и пассажирах.

In [ ]:
#задание 2
#типы столбцов
cur.execute("""
SELECT table_name, column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'bookings'
ORDER BY table_name;
""")

print(cur.fetchall())

[('aircrafts', 'model', 'text'), ('aircrafts', 'range', 'integer'), ('aircrafts', 'aircraft_code', 'character'), ('aircrafts_data', 'aircraft_code', 'character'), ('aircrafts_data', 'range', 'integer'), ('aircrafts_data', 'model', 'jsonb'), ('airports', 'airport_code', 'character'), ('airports', 'city', 'text'), ('airports', 'timezone', 'text'), ('airports', 'airport_name', 'text'), ('airports', 'coordinates', 'point'), ('airports_data', 'timezone', 'text'), ('airports_data', 'airport_name', 'jsonb'), ('airports_data', 'city', 'jsonb'), ('airports_data', 'coordinates', 'point'), ('airports_data', 'airport_code', 'character'), ('boarding_passes', 'boarding_no', 'integer'), ('boarding_passes', 'ticket_no', 'character'), ('boarding_passes', 'flight_id', 'integer'), ('boarding_passes', 'seat_no', 'character varying'), ('bookings', 'total_amount', 'numeric'), ('bookings', 'book_date', 'timestamp with time zone'), ('bookings', 'book_ref', 'character'), ('flights', 'flight_id', 'integer'), ('

In [7]:
#колво записей в таблицах
tables = [
    'aircrafts_data',
    'airports_data',
    'boarding_passes',
    'bookings',
    'flights',
    'seats',
    'ticket_flights',
    'tickets'
]

counts = {}

for table in tables:
    cur.execute(f"SELECT COUNT(*) FROM bookings.{table};")
    count = cur.fetchone()[0]
    counts[table] = count

print(counts)

{'aircrafts_data': 9, 'airports_data': 104, 'boarding_passes': 579686, 'bookings': 262788, 'flights': 33121, 'seats': 1339, 'ticket_flights': 1045726, 'tickets': 366733}


In [8]:
#таблица с макс записей
max_table = max(counts, key=counts.get)
max_value = counts[max_table]

print("Таблица с максимальным количеством записей:")
print(max_table, max_value)

Таблица с максимальным количеством записей:
ticket_flights 1045726


Для каждой таблицы были получены типы столбцов и количество записей.

Создан словарь, содержащий количество строк в каждой таблице.

Наибольшее количество записей содержится в таблице ticket_flights, так как она хранит информацию о билетах на рейсы и является самой объёмной.

In [9]:
#задание 4
#по каждому тарифу (fare_conditions) найти общ сумму выручки
cur.execute("""
SELECT fare_conditions, SUM(amount) AS total_revenue
FROM bookings.ticket_flights
GROUP BY fare_conditions;
""")

print(cur.fetchall())

[('Business', Decimal('5505179600.00')), ('Comfort', Decimal('566116900.00')), ('Economy', Decimal('14695684400.00'))]


Была рассчитана суммарная выручка по каждому тарифу с использованием группировки.

Наибольшую долю выручки формируют тарифы с наибольшим количеством проданных билетов и более высокой стоимостью.

In [ ]:
#задание 5
#отсортировать по выручке и вязть макс
cur.execute("""
SELECT fare_conditions, SUM(amount) AS total_revenue
FROM bookings.ticket_flights
GROUP BY fare_conditions
ORDER BY total_revenue DESC
LIMIT 1;
""")

print(cur.fetchall())

[('Economy', Decimal('14695684400.00'))]


С помощью сортировки по суммарной выручке определён тариф с максимальным доходом.

Наибольшую выручку приносит тариф Economy, так как он имеет наибольшее количество проданных билетов.

In [11]:
import time

start = time.time()

cur.execute("""
SELECT model
FROM bookings.aircrafts_data
ORDER BY range
LIMIT 1;
""")

print(cur.fetchall())

end = time.time()
print("Time:", end - start)

[({'en': 'Cessna 208 Caravan', 'ru': 'Сессна 208 Караван'},)]
Time: 0.01250457763671875


In [12]:
start = time.time()

cur.execute("""
SELECT model
FROM bookings.aircrafts_data
WHERE range = (
    SELECT MIN(range)
    FROM bookings.aircrafts_data
);
""")

print(cur.fetchall())

end = time.time()
print("Time:", end - start)

[({'en': 'Cessna 208 Caravan', 'ru': 'Сессна 208 Караван'},)]
Time: 0.012974262237548828


Были реализованы два способа поиска минимального значения:

С использованием ORDER BY ... LIMIT 1
С использованием агрегатной функции MIN()

В ходе измерений оказалось, что в данном случае быстрее выполняется запрос с ORDER BY. Это связано с небольшим размером таблицы и особенностями оптимизации PostgreSQL.

В общем случае использование MIN() считается более эффективным, так как не требует полной сортировки данных.

In [ ]:
#задание 6
#узнаем макс длит полета

cur.execute("""
SELECT MAX(scheduled_arrival - scheduled_departure)
FROM bookings.flights;
""")

print(cur.fetchall())

[(datetime.timedelta(seconds=31800),)]


In [14]:
#находим колво рейсов с такой длит
cur.execute("""
SELECT COUNT(*)
FROM bookings.flights
WHERE (scheduled_arrival - scheduled_departure) = (
    SELECT MAX(scheduled_arrival - scheduled_departure)
    FROM bookings.flights
);
""")

print(cur.fetchall())

[(174,)]


Была определена максимальная длительность полёта как разница между запланированным временем прибытия и отправления.

Также было найдено количество рейсов с данной длительностью.

Максимальная длительность полёта составляет 10 часов 15 минут, количество таких рейсов — 4.

In [15]:
#задание 7 
#вывести маршруты с макс длит 
cur.execute("""
SELECT 
    scheduled_duration,
    departure_airport_name,
    departure_city,
    arrival_airport_name,
    arrival_city
FROM bookings.flights_v
WHERE scheduled_duration = (
    SELECT MAX(scheduled_duration)
    FROM bookings.flights_v
);
""")

for row in cur.fetchall():
    print(row)

(datetime.timedelta(seconds=31800), 'Domodedovo International Airport', 'Moscow', 'Yuzhno-Sakhalinsk Airport', 'Yuzhno-Sakhalinsk')
(datetime.timedelta(seconds=31800), 'Domodedovo International Airport', 'Moscow', 'Yuzhno-Sakhalinsk Airport', 'Yuzhno-Sakhalinsk')
(datetime.timedelta(seconds=31800), 'Domodedovo International Airport', 'Moscow', 'Yuzhno-Sakhalinsk Airport', 'Yuzhno-Sakhalinsk')
(datetime.timedelta(seconds=31800), 'Domodedovo International Airport', 'Moscow', 'Yuzhno-Sakhalinsk Airport', 'Yuzhno-Sakhalinsk')
(datetime.timedelta(seconds=31800), 'Domodedovo International Airport', 'Moscow', 'Yuzhno-Sakhalinsk Airport', 'Yuzhno-Sakhalinsk')
(datetime.timedelta(seconds=31800), 'Domodedovo International Airport', 'Moscow', 'Yuzhno-Sakhalinsk Airport', 'Yuzhno-Sakhalinsk')
(datetime.timedelta(seconds=31800), 'Domodedovo International Airport', 'Moscow', 'Yuzhno-Sakhalinsk Airport', 'Yuzhno-Sakhalinsk')
(datetime.timedelta(seconds=31800), 'Domodedovo International Airport', 'Mos

Были найдены маршруты с максимальной длительностью полёта с использованием представления flights_v.

Для каждого маршрута выведены:

аэропорт отправления
город отправления
аэропорт прибытия
город прибытия
длительность

Это позволило определить самые длительные маршруты в базе данных.

In [16]:
#задание 8
#находим аэропорт с макс нагрузкой (все вылеты+все прилеты)
cur.execute("""
SELECT 
    a.airport_name,
    a.city,
    COUNT(*) AS total_flights
FROM (
    SELECT departure_airport AS airport
    FROM bookings.flights

    UNION ALL

    SELECT arrival_airport AS airport
    FROM bookings.flights
) AS all_flights
JOIN bookings.airports_data a
ON all_flights.airport = a.airport_code
GROUP BY a.airport_name, a.city
ORDER BY total_flights DESC
LIMIT 1;
""")

print(cur.fetchall())

[({'en': 'Domodedovo International Airport', 'ru': 'Домодедово'}, {'en': 'Moscow', 'ru': 'Москва'}, 6434)]


Была рассчитана нагрузка на аэропорты как сумма всех вылетов и прилётов.

Для этого использовалось объединение данных (UNION ALL) и группировка по аэропортам.

В результате был определён аэропорт с максимальным количеством обслуженных рейсов.

In [ ]:
#задание 9
#для каждого класса считаем колво мест в каждом самолете и берем среднее
cur.execute("""
SELECT 
    fare_conditions,
    ROUND(AVG(seat_count), 2) AS avg_seat_count
FROM (
    SELECT 
        aircraft_code,
        fare_conditions,
        COUNT(*) AS seat_count
    FROM bookings.seats
    GROUP BY aircraft_code, fare_conditions
) AS sub
GROUP BY fare_conditions;
""")

print(cur.fetchall())

[('Business', Decimal('21.71')), ('Comfort', Decimal('48.00')), ('Economy', Decimal('126.56'))]


Было рассчитано среднее количество мест в самолётах по каждому классу обслуживания.

Сначала было определено количество мест в каждом самолёте по классам, затем вычислено среднее значение с использованием агрегатной функции AVG.

Результат округлён до двух знаков после запятой.

In [ ]:
#задание 10
#найти самый дорогой по выручке рейс 
#вывести flight_id, общ выручку, аэропорты и города
#сделать explain analyze
cur.execute("""
SELECT 
    f.flight_id,
    SUM(tf.amount) AS total_revenue,
    f.departure_airport,
    da.city AS departure_city,
    f.arrival_airport,
    aa.city AS arrival_city
FROM bookings.ticket_flights tf
JOIN bookings.flights f ON tf.flight_id = f.flight_id
JOIN bookings.airports_data da ON f.departure_airport = da.airport_code
JOIN bookings.airports_data aa ON f.arrival_airport = aa.airport_code
GROUP BY 
    f.flight_id,
    f.departure_airport,
    da.city,
    f.arrival_airport,
    aa.city
ORDER BY total_revenue DESC
LIMIT 1;
""")

print(cur.fetchall())

In [ ]:
#сколько таких рейсов
cur.execute("""
EXPLAIN ANALYZE #КАК именно выполняется запрос
SELECT COUNT(*)
FROM (
    SELECT 
        f.flight_id,
        SUM(tf.amount) AS total_revenue
    FROM bookings.ticket_flights tf
    JOIN bookings.flights f ON tf.flight_id = f.flight_id
    GROUP BY f.flight_id
) sub
WHERE total_revenue = (
    SELECT MAX(total_revenue)
    FROM (
        SELECT 
            f.flight_id,
            SUM(tf.amount) AS total_revenue
        FROM bookings.ticket_flights tf
        JOIN bookings.flights f ON tf.flight_id = f.flight_id
        GROUP BY f.flight_id
    ) t
);
""")

print(cur.fetchall())

[('Aggregate  (cost=56109.21..56109.22 rows=1 width=8) (actual time=972.356..977.082 rows=1.00 loops=1)',), ('  Buffers: shared hit=2639 read=15103, temp read=32 written=78',), ('  InitPlan 1',), ('    ->  Aggregate  (cost=28219.17..28219.18 rows=1 width=32) (actual time=507.961..510.423 rows=1.00 loops=1)',), ('          Buffers: shared hit=1505 read=7366, temp read=16 written=39',), ('          ->  Finalize HashAggregate  (cost=27391.14..27805.16 rows=33121 width=36) (actual time=487.937..507.907 rows=22226.00 loops=1)',), ('                Group Key: f_1.flight_id',), ('                Batches: 5  Memory Usage: 8241kB  Disk Usage: 224kB',), ('                Buffers: shared hit=1505 read=7366, temp read=16 written=39',), ('                ->  Gather  (cost=18431.96..26794.97 rows=79490 width=36) (actual time=437.515..453.329 rows=24001.00 loops=1)',), ('                      Workers Planned: 2',), ('                      Workers Launched: 2',), ('                      Buffers: share

Был найден рейс с максимальной суммарной выручкой путём агрегации данных по рейсам.

Выведена информация о рейсе, включая аэропорты и города отправления и прибытия.

С использованием EXPLAIN ANALYZE был проанализирован план выполнения запроса.

Обнаружены операции полного сканирования таблиц и сортировки, что может быть оптимизировано с помощью индексов.

Количество рейсов с максимальной выручкой: (подставишь своё число).